# Case Study: End-to-End ML Project - Energy Consumption Forecasting

## 1. Problem Definition and Project Setup

In this case study, we'll develop a machine learning system to forecast hourly electricity consumption for a utility company. Accurate forecasting helps with resource planning, pricing optimization, and grid management.

```python
# Project goal definition
"""
Project: Energy Consumption Forecasting
Goal: Predict hourly electricity consumption for the next 24-48 hours
Business Impact:
- Optimize power generation resources (estimated 5-10% cost savings)
- Improve grid stability planning
- Enable dynamic pricing models
- Reduce carbon footprint through optimized energy generation

Success Metrics:
- RMSE < 5% of peak consumption
- MAPE < 10%
- Reliable predictions during peak demand periods
"""
```

### Setting Up the Project Structure

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set plot styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")

# Create project directories
directories = [
    'data/raw',
    'data/processed',
    'models',
    'visualizations',
    'reports'
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"Created directory: {directory}")

## 2. Data Collection and Exploration

### Obtaining Energy Consumption Data

In [ ]:
# For this example, we'll use open data from PJM Interconnection
# This contains hourly power consumption data

# Download data (in real project, you might use an API or direct database connection)
import urllib.request

# Correct URL for hourly load data
# url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/household_power_consumption_days.csv"
url = "https://raw.githubusercontent.com/Talish-wiz/Improved-Home-Electrcity-Forecasting-Streamlit/refs/heads/master/household_power_consumption_days.csv"
data_path = "data/raw/energy_consumption.csv"

urllib.request.urlretrieve(url, data_path)
print(f"Downloaded data to {data_path}")

# Load the data
df = pd.read_csv(data_path)

# Convert date strings to datetime
df['datetime'] = pd.to_datetime(df['datetime'], format='%Y-%m-%d')
df.set_index('datetime', inplace=True)

# Add sub_metering_4 if not present (this was mentioned in the CSV structure)
if 'sub_metering_4' not in df.columns and 'Sub_metering_3' in df.columns:
    # Calculate sub_metering_4 as the difference between total and other sub meters
    df['sub_metering_4'] = df['Global_active_power'] * 1000 / 60 - (df['Sub_metering_1'] + df['Sub_metering_2'] + df['Sub_metering_3'])

print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

# Basic information
print("\nData types and missing values:")
print(df.info())

# Summary statistics
print("\nSummary statistics:")
print(df.describe())

### Exploratory Data Analysis

In [ ]:
# Time series visualization
plt.figure(figsize=(15, 7))
plt.plot(df.index, df['Global_active_power'], linewidth=1)
plt.title('Global Active Power Over Time')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kilowatts)')
plt.tight_layout()
plt.savefig('visualizations/power_over_time.png')
plt.show()

# Check for seasonality
# Daily seasonality
hourly_data = df.copy()  # Data is already daily in this dataset
hourly_avg = hourly_data.groupby(hourly_data.index.hour).mean() if 'hour' in hourly_data.index.names else hourly_data

plt.figure(figsize=(12, 6))
plt.plot(range(24), [hourly_avg['Global_active_power'].mean()] * 24, marker='o')  # Placeholder since we don't have hourly data
plt.title('Average Power Consumption by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Average Global Active Power (kilowatts)')
plt.grid(True)
plt.savefig('visualizations/daily_seasonality.png')
plt.show()

# Weekly seasonality
daily_data = df.copy()  # Data is already daily
weekly_avg = daily_data.groupby(daily_data.index.dayofweek).mean() if 'dayofweek' in daily_data.index.names else pd.DataFrame()

plt.figure(figsize=(12, 6))
plt.plot(range(7), [daily_data['Global_active_power'].mean()] * 7, marker='o')  # Placeholder
plt.title('Average Power Consumption by Day of Week')
plt.xlabel('Day of Week (0=Monday, 6=Sunday)')
plt.ylabel('Average Global Active Power (kilowatts)')
plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
plt.grid(True)
plt.savefig('visualizations/weekly_seasonality.png')
plt.show()

# Monthly seasonality
# monthly_data = df.resample('M').mean()
# FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
monthly_data = df.resample('ME').mean()
monthly_avg = monthly_data.groupby(monthly_data.index.month).mean()

plt.figure(figsize=(12, 6))
plt.plot(monthly_avg.index, monthly_avg['Global_active_power'], marker='o')
plt.title('Average Power Consumption by Month')
plt.xlabel('Month')
plt.ylabel('Average Global Active Power (kilowatts)')
plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.grid(True)
plt.savefig('visualizations/monthly_seasonality.png')
plt.show()

# Check for correlations between features
correlation_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlations')
plt.tight_layout()
plt.savefig('visualizations/feature_correlations.png')
plt.show()

## 3. Data Preprocessing and Feature Engineering

### Data Cleaning

In [ ]:
# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Fill missing values using forward fill (assuming time series continuity)
# df_clean = df.fillna(method='ffill')
# FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
df_clean = df.ffill()

# Check for outliers using IQR
Q1 = df_clean['Global_active_power'].quantile(0.25)
Q3 = df_clean['Global_active_power'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['Global_active_power'] < lower_bound) | 
                   (df_clean['Global_active_power'] > upper_bound)]

print(f"\nNumber of outliers detected: {len(outliers)}")

# Visualize outliers
plt.figure(figsize=(12, 6))
plt.scatter(df_clean.index, df_clean['Global_active_power'], s=2, label='Data')
plt.scatter(outliers.index, outliers['Global_active_power'], color='red', s=5, label='Outliers')
plt.title('Power Consumption with Outliers Highlighted')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kilowatts)')
plt.legend()
plt.savefig('visualizations/outliers.png')
plt.show()

# For this tutorial, we'll cap outliers instead of removing them
df_clean['Global_active_power'] = df_clean['Global_active_power'].clip(lower_bound, upper_bound)
print("\nOutliers capped at IQR boundaries")

### Feature Engineering

In [ ]:
# Create time-based features
df_features = df_clean.copy()

# Date-based features
df_features['hour'] = df_features.index.hour
df_features['dayofweek'] = df_features.index.dayofweek
df_features['quarter'] = df_features.index.quarter
df_features['month'] = df_features.index.month
df_features['year'] = df_features.index.year
df_features['dayofyear'] = df_features.index.dayofyear

# Create cyclical features for hour, day of week and month
# This preserves the cyclic nature of these features
df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour']/24)
df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour']/24)

df_features['dow_sin'] = np.sin(2 * np.pi * df_features['dayofweek']/7)
df_features['dow_cos'] = np.cos(2 * np.pi * df_features['dayofweek']/7)

df_features['month_sin'] = np.sin(2 * np.pi * df_features['month']/12)
df_features['month_cos'] = np.cos(2 * np.pi * df_features['month']/12)

# Is weekend feature
df_features['is_weekend'] = df_features['dayofweek'].isin([5, 6]).astype(int)

# Lag features (previous days consumption)
for lag in [1, 2, 3, 7, 14, 30]:  # days since we're working with daily data
    df_features[f'lag_{lag}d'] = df_features['Global_active_power'].shift(lag)

# Rolling window features
for window in [3, 7, 14, 30]:  # days
    df_features[f'rolling_mean_{window}d'] = df_features['Global_active_power'].rolling(window=window).mean()
    df_features[f'rolling_std_{window}d'] = df_features['Global_active_power'].rolling(window=window).std()

# Drop NaN values created by lag and rolling features
df_features = df_features.dropna()

print("\nEngineered feature dataset shape:", df_features.shape)
print("\nEngineered features:")
print(df_features.columns.tolist())

# Display a sample of the engineered features
print("\nSample of engineered features:")
print(df_features.head())

## 4. Feature Selection and Dataset Preparation

In [ ]:
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# Define target variable and features
target = 'Global_active_power'
exclude_columns = ['Global_reactive_power', 'Voltage', 'Global_intensity', 
                   'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'sub_metering_4']

features = [col for col in df_features.columns if col != target and col not in exclude_columns]

X = df_features[features]
y = df_features[target]

# Split data chronologically (important for time series)
# We'll use the last 20% as the test set
split_idx = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training data: {X_train.shape[0]} samples")
print(f"Testing data: {X_test.shape[0]} samples")

# Standardize numerical features
scaler = StandardScaler()
numerical_features = [col for col in X_train.columns if X_train[col].dtype != 'object' 
                     and col not in ['hour', 'dayofweek', 'month', 'year', 'is_weekend']]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

# Save the processed datasets
X_train_scaled.to_csv('data/processed/X_train.csv')
X_test_scaled.to_csv('data/processed/X_test.csv')
y_train.to_csv('data/processed/y_train.csv')
y_test.to_csv('data/processed/y_test.csv')

print("\nPreprocessed data saved to 'data/processed/' directory")

## 5. Model Development and Evaluation

### Baseline Models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb

# Define evaluation metrics function
def evaluate_model(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100  # Convert to percentage
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{model_name} Performance:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}%")
    print(f"R²: {r2:.4f}")
    
    return {
        'model_name': model_name,
        'rmse': rmse,
        'mape': mape,
        'r2': r2
    }

# Dictionary to store model results
results = {}

# 1. Naive Forecast (previous day's value)
y_pred_naive = X_test['lag_1d']  # Using the 1-day lag feature as naive forecast
results['Naive Forecast'] = evaluate_model('Naive Forecast', y_test, y_pred_naive)

# 2. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
results['Linear Regression'] = evaluate_model('Linear Regression', y_test, y_pred_lr)

# 3. Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
results['Random Forest'] = evaluate_model('Random Forest', y_test, y_pred_rf)

# 4. Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train_scaled, y_train)
y_pred_gb = gb_model.predict(X_test_scaled)
results['Gradient Boosting'] = evaluate_model('Gradient Boosting', y_test, y_pred_gb)

# 5. XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
results['XGBoost'] = evaluate_model('XGBoost', y_test, y_pred_xgb)

# 6. LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
lgb_model.fit(X_train_scaled, y_train)
y_pred_lgb = lgb_model.predict(X_test_scaled)
results['LightGBM'] = evaluate_model('LightGBM', y_test, y_pred_lgb)

# Create results dataframe for visualization
results_df = pd.DataFrame([
    results['Naive Forecast'],
    results['Linear Regression'],
    results['Random Forest'],
    results['Gradient Boosting'],
    results['XGBoost'],
    results['LightGBM']
])

# Visualize model comparison
plt.figure(figsize=(12, 8))

# RMSE Comparison
plt.subplot(2, 1, 1)
plt.barh(results_df['model_name'], results_df['rmse'])
plt.title('Model Comparison - RMSE (lower is better)')
plt.xlabel('RMSE')
plt.xlim(left=0)
for i, v in enumerate(results_df['rmse']):
    plt.text(v + 0.01, i, f"{v:.3f}")

# MAPE Comparison
plt.subplot(2, 1, 2)
plt.barh(results_df['model_name'], results_df['mape'])
plt.title('Model Comparison - MAPE (lower is better)')
plt.xlabel('MAPE (%)')
plt.xlim(left=0)
for i, v in enumerate(results_df['mape']):
    plt.text(v + 0.01, i, f"{v:.1f}%")

plt.tight_layout()
plt.savefig('visualizations/model_comparison.png')
plt.show()

# Select the best performing model (based on results)
best_model = xgb_model  # Update this based on actual results
best_model_name = 'XGBoost'  # Update this based on actual results

### Visualizing Predictions

In [ ]:
# Plot the actual vs predicted values for the best model
plt.figure(figsize=(15, 7))

# Get the appropriate predictions
if best_model_name == 'XGBoost':
    y_pred = y_pred_xgb
elif best_model_name == 'LightGBM':
    y_pred = y_pred_lgb
elif best_model_name == 'Random Forest':
    y_pred = y_pred_rf
elif best_model_name == 'Gradient Boosting':
    y_pred = y_pred_gb
elif best_model_name == 'Linear Regression':
    y_pred = y_pred_lr
else:
    y_pred = y_pred_naive

# Plot
plt.plot(y_test.index, y_test.values, label='Actual', linewidth=2)
plt.plot(y_test.index, y_pred, label=f'Predicted ({best_model_name})', linewidth=2, alpha=0.8)
plt.title('Actual vs Predicted Power Consumption')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kilowatts)')
plt.legend()
plt.grid(True)
plt.savefig('visualizations/actual_vs_predicted.png')
plt.show()

# Plot residuals
residuals = y_test - y_pred
plt.figure(figsize=(15, 7))
plt.plot(y_test.index, residuals, color='red', alpha=0.7)
plt.axhline(y=0, color='black', linestyle='--')
plt.title('Prediction Residuals Over Time')
plt.xlabel('Date')
plt.ylabel('Residual (Actual - Predicted)')
plt.grid(True)
plt.savefig('visualizations/residuals.png')
plt.show()

# Histogram of residuals
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=30, alpha=0.7, color='blue')
plt.axvline(x=0, color='red', linestyle='--')
plt.title('Distribution of Residuals')
plt.xlabel('Residual Value')
plt.ylabel('Frequency')
plt.grid(True)
plt.savefig('visualizations/residuals_histogram.png')
plt.show()

### Feature Importance

In [ ]:
# For tree-based models, extract feature importance
if best_model_name in ['Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM']:
    if best_model_name == 'XGBoost':
        importances = best_model.feature_importances_
    else:
        importances = best_model.feature_importances_
    
    # Create a DataFrame for easier visualization
    feature_importance = pd.DataFrame({
        'Feature': X_train_scaled.columns,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    # Plot feature importance
    plt.figure(figsize=(12, 8))
    plt.barh(feature_importance['Feature'][:15], feature_importance['Importance'][:15])
    plt.title(f'Top 15 Feature Importances ({best_model_name})')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.gca().invert_yaxis()  # Display highest importance at the top
    plt.tight_layout()
    plt.savefig('visualizations/feature_importance.png')
    plt.show()
    
    print(f"\nTop 10 Most Important Features ({best_model_name}):")
    print(feature_importance.head(10))

## 6. Model Optimization

### Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# Define hyperparameter search space for XGBoost
param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'min_child_weight': [1, 2, 3, 4]
}

# Use TimeSeriesSplit for time series data
tscv = TimeSeriesSplit(n_splits=5)

# Random search for hyperparameters
random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(objective='reg:squarederror', random_state=42),
    param_distributions=param_dist,
    n_iter=20,  # Reduced from 50 to speed up example
    scoring='neg_mean_squared_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit random search
print("\nPerforming hyperparameter tuning. This may take some time...")
random_search.fit(X_train_scaled, y_train)

# Best parameters and score
print("\nBest hyperparameters:")
print(random_search.best_params_)
best_score = np.sqrt(-random_search.best_score_)
print(f"Best RMSE: {best_score:.4f}")

# Create optimized model with best parameters
tuned_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, **random_search.best_params_)

# Train on full training set
tuned_model.fit(X_train_scaled, y_train)

# Evaluate tuned model
y_pred_tuned = tuned_model.predict(X_test_scaled)
tuned_results = evaluate_model(f'Tuned {best_model_name}', y_test, y_pred_tuned)

# Compare with previous best model
plt.figure(figsize=(10, 6))
comparison = pd.DataFrame([
    results[best_model_name],
    tuned_results
])
comparison[['rmse', 'mape']].plot(kind='bar', figsize=(10, 6))
plt.title('Model Performance Before and After Tuning')
plt.ylabel('Value')
plt.xticks([0, 1], [best_model_name, f'Tuned {best_model_name}'], rotation=0)
plt.grid(True, axis='y')
plt.legend(['RMSE', 'MAPE (%)'])

# Add value labels on bars
for i, metric in enumerate(['rmse', 'mape']):
    for j, model in enumerate([best_model_name, f'Tuned {best_model_name}']):
        if j == 0:
            value = results[best_model_name][metric]
        else:
            value = tuned_results[metric]
        plt.text(j - 0.1 + i*0.2, value + 0.1, f"{value:.2f}", rotation=0)

plt.tight_layout()
plt.savefig('visualizations/tuning_comparison.png')
plt.show()

# Update best model to tuned version
best_model = tuned_model

## 7. Model Deployment

### Model Serialization

In [ ]:
import joblib
import os

# Save the trained model and scaler
model_path = os.path.join('models', 'energy_forecast_model.pkl')
scaler_path = os.path.join('models', 'scaler.pkl')

joblib.dump(best_model, model_path)
joblib.dump(scaler, scaler_path)

print(f"\nModel saved to {model_path}")
print(f"Scaler saved to {scaler_path}")

### Creating a Prediction Function

In [ ]:
def make_forecast(days_to_predict=7, model_path='models/energy_forecast_model.pkl', 
                  scaler_path='models/scaler.pkl'):
    """
    Generate energy consumption forecast for the specified number of days
    
    Parameters:
    -----------
    days_to_predict : int
        Number of days to forecast into the future
    model_path : str
        Path to the saved model file
    scaler_path : str
        Path to the saved scaler file
        
    Returns:
    --------
    pd.DataFrame
        DataFrame containing the forecasted values with timestamps
    """
    # Load model and scaler
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    
    # Get the last available data point
    latest_data = df_features.iloc[-1]
    
    # Initialize results storage
    forecast_dates = []
    forecast_values = []
    
    # Get the last timestamp from the data
    last_timestamp = df_features.index[-1]
    
    # Create a copy of the latest data for making predictions
    current_data = latest_data.copy()
    
    # Generate predictions for each day
    for i in range(1, days_to_predict + 1):
        # Calculate the next timestamp
        next_timestamp = last_timestamp + pd.Timedelta(days=i)
        forecast_dates.append(next_timestamp)
        
        # Update time features for the next day
        current_data['hour'] = next_timestamp.hour
        current_data['dayofweek'] = next_timestamp.dayofweek
        current_data['quarter'] = next_timestamp.quarter
        current_data['month'] = next_timestamp.month
        current_data['year'] = next_timestamp.year
        current_data['dayofyear'] = next_timestamp.dayofyear
        current_data['is_weekend'] = 1 if next_timestamp.dayofweek >= 5 else 0
        
        # Update cyclical features
        current_data['hour_sin'] = np.sin(2 * np.pi * next_timestamp.hour/24)
        current_data['hour_cos'] = np.cos(2 * np.pi * next_timestamp.hour/24)
        current_data['dow_sin'] = np.sin(2 * np.pi * next_timestamp.dayofweek/7)
        current_data['dow_cos'] = np.cos(2 * np.pi * next_timestamp.dayofweek/7)
        current_data['month_sin'] = np.sin(2 * np.pi * next_timestamp.month/12)
        current_data['month_cos'] = np.cos(2 * np.pi * next_timestamp.month/12)
        
        # Extract features needed for prediction
        feature_vector = pd.DataFrame([current_data[X_train_scaled.columns].values], 
                                     columns=X_train_scaled.columns)
        
        # Apply scaling to appropriate columns
        feature_vector[numerical_features] = scaler.transform(feature_vector[numerical_features])
        
        # Make prediction
        prediction = model.predict(feature_vector)[0]
        forecast_values.append(prediction)
        
        # Update lag features for next iteration
        for lag in range(30, 0, -1):  # Update from oldest to newest
            lag_key = f'lag_{lag}d'
            if lag_key in current_data.index:
                if lag > 1:
                    prev_lag = f'lag_{lag-1}d'
                    if prev_lag in current_data.index:
                        current_data[lag_key] = current_data[prev_lag]
                else:
                    # lag_1d gets the latest prediction
                    current_data[f'lag_1d'] = prediction
    
        # Update rolling means (simplified)
        if f'rolling_mean_30d' in current_data.index:
            lag_values = [current_data[f'lag_{i}d'] for i in [1, 3, 7, 14, 30] 
                         if f'lag_{i}d' in current_data.index]
            if lag_values:
                current_data[f'rolling_mean_30d'] = sum(lag_values) / len(lag_values)
    
    # Create forecast DataFrame
    forecast_df = pd.DataFrame({
        'timestamp': forecast_dates,
        'forecasted_consumption': forecast_values
    })
    
    forecast_df.set_index('timestamp', inplace=True)
    
    return forecast_df

# Generate a 14-day forecast
forecast = make_forecast(days_to_predict=14)

# Plot the forecast
plt.figure(figsize=(15, 7))

# Plot the last 30 days of actual data
last_month = df[target].iloc[-30:]
plt.plot(last_month.index, last_month.values, label='Historical Data', color='blue')

# Plot the forecast
plt.plot(forecast.index, forecast['forecasted_consumption'], label='Forecast', color='red', linestyle='--')

# Add a vertical line to separate historical data from forecast
forecast_start = forecast.index[0]
plt.axvline(forecast_start, color='gray', linestyle='-', alpha=0.7)
plt.text(forecast_start, plt.ylim()[1]*0.9, 'Forecast Start', 
         rotation=90, verticalalignment='top')

plt.title('Energy Consumption Forecast (Next 14 Days)')
plt.xlabel('Date')
plt.ylabel('Global Active Power (kilowatts)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('visualizations/forecast_14d.png')
plt.show()

print("\nForecast for the next 14 days:")
print(forecast)

### Creating a Simple API

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/forecast', methods=['GET'])
def api_forecast():
    """API endpoint to get energy consumption forecast"""
    try:
        # Get days parameter (default to 7 days)
        days = request.args.get('days', default=7, type=int)
        
        # Limit forecast to reasonable range
        if days < 1:
            return jsonify({'error': 'Days must be at least 1'}), 400
        if days > 30:  # 30 days max
            return jsonify({'error': 'Forecast limited to maximum of 30 days'}), 400
        
        # Generate forecast
        forecast = make_forecast(days_to_predict=days)
        
        # Convert to dictionary for JSON response
        response = {
            'forecast': [
                {
                    'timestamp': timestamp.strftime('%Y-%m-%d'),
                    'consumption': value
                }
                for timestamp, value in zip(forecast.index, forecast['forecasted_consumption'])
            ],
            'units': 'kilowatts',
            'model_version': '1.0'
        }
        
        return jsonify(response)
    
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Example of starting the API (in production use a WSGI server)
if __name__ == '__main__':
    # app.run(debug=True, port=5000)
    app.run(debug=True, use_reloader=False, port=5000)

# """
# Example API request:
# http://127.0.0.1:5000/forecast
# http://localhost:5000/forecast?days=7
# """

## 8. Model Monitoring

In [ ]:
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Define monitoring functions
def simulate_production_data(days=30, model=None, scaler=None):
    """
    Simulate how the model would perform in production over time
    
    Parameters:
    -----------
    days : int
        Number of days to simulate
    model : object
        Trained forecasting model
    scaler : object
        Fitted scaler for data normalization
        
    Returns:
    --------
    pd.DataFrame
        DataFrame containing performance metrics over time
    """
    if model is None:
        model = joblib.load('models/energy_forecast_model.pkl')
    if scaler is None:
        scaler = joblib.load('models/scaler.pkl')
    
    # Start from the beginning of test data
    start_date = y_test.index[0]
    
    # Storage for results
    results = []
    
    # Sliding window simulation over the test period
    for day in range(min(days, len(y_test))):
        # Define time window
        current_date = start_date + pd.Timedelta(days=day)
        
        # Get actual data for this day
        actual_value = y_test.loc[current_date] if current_date in y_test.index else None
        if actual_value is None:  # Skip days without actual data
            continue
        
        # Get feature data for this day
        if current_date in X_test_scaled.index:
            features = X_test_scaled.loc[current_date:current_date]
            
            # Make prediction
            prediction = model.predict(features)[0]
            
            # Calculate metrics for this point
            error = actual_value - prediction
            abs_error = abs(error)
            pct_error = abs(error) / actual_value * 100
            
            # Add some simulated drift over time
            drift_factor = 1 + (day / (days * 5))  # Gradual increase in error
            
            # Store results
            results.append({
                'date': current_date.date(),
                'rmse': abs_error * drift_factor,  # Using absolute error as a proxy for RMSE with single point
                'mape': pct_error * drift_factor,
                'prediction': prediction,
                'actual': actual_value
            })
    
    return pd.DataFrame(results)

# Generate monitoring data
monitoring_data = simulate_production_data(days=30)

# Visualize monitoring metrics
plt.figure(figsize=(15, 10))

# Error over time
plt.subplot(2, 2, 1)
plt.plot(monitoring_data['date'], monitoring_data['rmse'], marker='o', color='blue')
plt.axhline(y=5.0, color='r', linestyle='--', alpha=0.7, label='Alert Threshold')
plt.title('Error Over Time')
plt.ylabel('Absolute Error')
plt.grid(True, alpha=0.3)
plt.legend()

# MAPE over time
plt.subplot(2, 2, 2)
plt.plot(monitoring_data['date'], monitoring_data['mape'], marker='o', color='green')
plt.axhline(y=10.0, color='r', linestyle='--', alpha=0.7, label='Alert Threshold')
plt.title('Percentage Error Over Time')
plt.ylabel('Error (%)')
plt.grid(True, alpha=0.3)
plt.legend()

# Prediction vs Actual
plt.subplot(2, 1, 2)
plt.plot(monitoring_data['date'], monitoring_data['prediction'], marker='o', label='Prediction')
plt.plot(monitoring_data['date'], monitoring_data['actual'], marker='x', label='Actual')
plt.title('Prediction vs Actual')
plt.ylabel('Power Consumption (kW)')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig('visualizations/model_monitoring.png')
plt.show()

# Check for performance degradation
alert_threshold_rmse = 5.0
alert_threshold_mape = 10.0

alerts = monitoring_data[
    (monitoring_data['rmse'] > alert_threshold_rmse) | 
    (monitoring_data['mape'] > alert_threshold_mape)
]

if not alerts.empty:
    print("\nPERFORMANCE ALERTS DETECTED:")
    print(alerts[['date', 'rmse', 'mape']])
    print("\nRecommendation: Model retraining may be needed.")
else:
    print("\nNo performance alerts detected. Model is performing within acceptable thresholds.")

## 9. Business Impact Analysis

In [ ]:
# def calculate_business_impact():
#     """
#     Calculate the business impact of the forecasting model
#     """
#     # Assumptions
#     avg_daily_consumption = y_test.mean()  # kW
#     electricity_cost_per_kwh = 0.15  # $
#     avg_error_without_model = 0.20  # 20% error without model
#     avg_error_with_model = tuned_results['mape'] / 100  # Our model's error rate
    
#     # Daily calculations
#     daily_cost = avg_daily_consumption * 24 * electricity_cost_per_kwh  # $
    
#     # Monthly calculations
#     monthly_consumption = avg_daily_consumption * 30 * 24  # kWh
#     monthly_cost = daily_cost * 30  # $
    
#     # Calculate potential savings
#     # Assumption: Better forecasting reduces waste by the difference in error rates
#     error_improvement = avg_error_without_model - avg_error_with_model
#     monthly_savings = monthly_cost * error_improvement
#     annual_savings = monthly_savings * 12
    
#     # ROI calculation
#     model_development_cost = 50000  # $ (hypothetical)
#     model_maintenance_cost = 1000  # $ per month (hypothetical)
#     annual_maintenance_cost = model_maintenance_cost * 12
    
#     first_year_roi = (annual_savings - model_development_cost - annual_maintenance_cost) / (model_development_cost + annual_maintenance_cost) * 100
#     subsequent_years_roi = (annual_savings - annual_maintenance_cost) / annual_maintenance_cost * 100
    
#     # Carbon footprint reduction
#     carbon_intensity = 0.5  # kg CO2 per kWh (average US grid)
#     monthly_carbon_savings = monthly_consumption * error_improvement * carbon_intensity  # kg CO2
#     annual_carbon_savings = monthly_carbon_savings * 12  # kg CO2
    
#     return {
#         'avg_daily_consumption': avg_daily_consumption,
#         'monthly_consumption': monthly_consumption,
#         'monthly_cost': monthly_cost,
#         'error_improvement': error_improvement * 100,  # to percentage
#         'monthly_savings': monthly_savings,
#         'annual_savings': annual_savings,
#         'first_year_roi': first_year_roi,
#         'subsequent_years_roi': subsequent_years_roi,
#         'annual_carbon_savings': annual_carbon_savings
#     }

# # Calculate business impact
# impact = calculate_business_impact()

# # Display results
# print("\nBusiness Impact Analysis:")
# print(f"Average daily consumption: {impact['avg_daily_consumption']:.2f} kW")
# print(f"Monthly electricity cost: ${impact['monthly_cost']:,.2f}")
# print(f"Forecasting error improvement: {impact['error_improvement']:.2f}%")
# print(f"Monthly cost savings: ${impact['monthly_savings']:,.2f}")
# print(f"Annual cost savings: ${impact['annual_savings']:,.2f}")
# print(f"First year ROI: {impact['first_year_roi']:.2f}%")
# print(f"Subsequent years ROI: {impact['subsequent_years_roi']:.2f}%")
# print(f"Annual carbon emission reduction: {impact['annual_carbon_savings']:,.2f} kg CO2")

# # Visualize business impact
# plt.figure(figsize=(15, 8))

# # Cost savings
# plt.subplot(2, 2, 1)
# savings = [impact['annual_savings'] for _ in range(5)]
# cumulative_savings = [sum(savings[:i+1]) for i in range(5)]
# plt.bar(range(1, 6), savings, alpha=0.7, label='Annual Savings')
# plt.plot(range(1, 6), cumulative_savings, 'ro-', label='Cumulative Savings')
# plt.title('Projected Annual Savings')
# plt.xlabel('Year')
# plt.ylabel('Savings ($)')
# plt.xticks(range(1, 6))
# plt.grid(True, alpha=0.3)
# plt.legend()

# # ROI
# plt.subplot(2, 2, 2)
# roi_values = [impact['first_year_roi']] + [impact['subsequent_years_roi'] for _ in range(4)]
# plt.bar(range(1, 6), roi_values)
# plt.title('Return on Investment')
# plt.xlabel('Year')
# plt.ylabel('ROI (%)')
# plt.xticks(range(1, 6))
# plt.grid(True, alpha=0.3)

# # Carbon savings
# plt.subplot(2, 2, 3)
# carbon_savings = [impact['annual_carbon_savings'] for _ in range(5)]
# cumulative_carbon = [sum(carbon_savings[:i+1]) for i in range(5)]
# plt.bar(range(1, 6), carbon_savings, color='green', alpha=0.7, label='Annual Reduction')
# plt.plot(range(1, 6), cumulative_carbon, 'go-', label='Cumulative Reduction')
# plt.title('Carbon Emission Reduction')
# plt.xlabel('Year')
# plt.ylabel('CO₂ Reduction (kg)')
# plt.xticks(range(1, 6))
# plt.grid(True, alpha=0.3)
# plt.legend()

# # Error reduction
# plt.subplot(2, 2, 4)
# plt.pie([impact['error_improvement'], 100 - impact['error_improvement']], 
#         labels=['Error Reduction', 'Remaining Error'],
#         autopct='%1.1f%%',
#         colors=['lightgreen', 'lightgray'],
#         startangle=90)
# plt.title('Forecasting Error Reduction')

# plt.tight_layout()
# plt.savefig('visualizations/business_impact.png')
# plt.show()

```
UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.tight_layout()

UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  plt.savefig('visualizations/business_impact.png')

UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Arial.
  fig.canvas.print_figure(bytes_io, **kw)
```

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib as mpl

# Ensure Unicode math text rendering
mpl.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'  # Font with full Unicode support

# Ensure 'visualizations' directory exists
os.makedirs("visualizations", exist_ok=True)

def calculate_business_impact():
    """
    Calculate the business impact of the forecasting model
    """
    # Assumptions
    avg_daily_consumption = y_test.mean()  # kW
    electricity_cost_per_kwh = 0.15  # $
    avg_error_without_model = 0.20  # 20% error without model
    avg_error_with_model = tuned_results['mape'] / 100  # Our model's error rate
    
    # Daily and monthly cost calculations
    daily_cost = avg_daily_consumption * 24 * electricity_cost_per_kwh
    monthly_consumption = avg_daily_consumption * 30 * 24
    monthly_cost = daily_cost * 30
    
    # Forecasting savings
    error_improvement = avg_error_without_model - avg_error_with_model
    monthly_savings = monthly_cost * error_improvement
    annual_savings = monthly_savings * 12
    
    # ROI
    model_development_cost = 50000
    model_maintenance_cost = 1000
    annual_maintenance_cost = model_maintenance_cost * 12

    first_year_roi = ((annual_savings - model_development_cost - annual_maintenance_cost) /
                      (model_development_cost + annual_maintenance_cost)) * 100
    subsequent_years_roi = ((annual_savings - annual_maintenance_cost) / 
                            annual_maintenance_cost) * 100

    # CO2 savings
    carbon_intensity = 0.5  # kg CO₂ per kWh
    monthly_carbon_savings = monthly_consumption * error_improvement * carbon_intensity
    annual_carbon_savings = monthly_carbon_savings * 12

    return {
        'avg_daily_consumption': avg_daily_consumption,
        'monthly_consumption': monthly_consumption,
        'monthly_cost': monthly_cost,
        'error_improvement': error_improvement * 100,
        'monthly_savings': monthly_savings,
        'annual_savings': annual_savings,
        'first_year_roi': first_year_roi,
        'subsequent_years_roi': subsequent_years_roi,
        'annual_carbon_savings': annual_carbon_savings
    }

# Run analysis
impact = calculate_business_impact()

# Output summary
print("\nBusiness Impact Analysis:")
print(f"Average daily consumption: {impact['avg_daily_consumption']:.2f} kW")
print(f"Monthly electricity cost: ${impact['monthly_cost']:,.2f}")
print(f"Forecasting error improvement: {impact['error_improvement']:.2f}%")
print(f"Monthly cost savings: ${impact['monthly_savings']:,.2f}")
print(f"Annual cost savings: ${impact['annual_savings']:,.2f}")
print(f"First year ROI: {impact['first_year_roi']:.2f}%")
print(f"Subsequent years ROI: {impact['subsequent_years_roi']:.2f}%")
print(f"Annual carbon emission reduction: {impact['annual_carbon_savings']:,.2f} kg CO₂")

# Plotting
plt.figure(figsize=(15, 8))

# Annual savings bar + cumulative line
plt.subplot(2, 2, 1)
years = range(1, 6)
savings = [impact['annual_savings']] * 5
cumulative_savings = [sum(savings[:i+1]) for i in range(5)]
plt.bar(years, savings, alpha=0.7, label='Annual Savings')
plt.plot(years, cumulative_savings, 'ro-', label='Cumulative Savings')
plt.title('Projected Annual Savings')
plt.xlabel('Year')
plt.ylabel('Savings ($)')
plt.xticks(years)
plt.grid(True, alpha=0.3)
plt.legend()

# ROI per year
plt.subplot(2, 2, 2)
roi_values = [impact['first_year_roi']] + [impact['subsequent_years_roi']] * 4
plt.bar(years, roi_values, color='skyblue')
plt.title('Return on Investment')
plt.xlabel('Year')
plt.ylabel('ROI (%)')
plt.xticks(years)
plt.grid(True, alpha=0.3)

# CO₂ savings
plt.subplot(2, 2, 3)
carbon_savings = [impact['annual_carbon_savings']] * 5
cumulative_carbon = [sum(carbon_savings[:i+1]) for i in range(5)]
plt.bar(years, carbon_savings, color='green', alpha=0.7, label='Annual Reduction')
plt.plot(years, cumulative_carbon, 'go-', label='Cumulative Reduction')
plt.title(r'Carbon Emission Reduction')
plt.xlabel('Year')
plt.ylabel(r'CO$_2$ Reduction (kg)')
plt.xticks(years)
plt.grid(True, alpha=0.3)
plt.legend()

# Forecasting error pie
plt.subplot(2, 2, 4)
plt.pie(
    [impact['error_improvement'], 100 - impact['error_improvement']],
    labels=['Error Reduction', 'Remaining Error'],
    autopct='%1.1f%%',
    colors=['lightgreen', 'lightgray'],
    startangle=90
)
plt.title('Forecasting Error Reduction')

plt.tight_layout()
plt.savefig('visualizations/business_impact.png', bbox_inches='tight')
plt.show()

## 10. Executive Summary and Documentation

```python
# Generate an executive summary in markdown
executive_summary = f"""
# Energy Consumption Forecasting Project: Executive Summary

## Project Overview
This project developed a machine learning system to forecast daily electricity consumption
for a utility company. The model enables more accurate resource planning, price optimization,
and grid management.

## Key Results

### Model Performance
- **RMSE**: {tuned_results['rmse']:.2f} kilowatts
- **MAPE**: {tuned_results['mape']:.2f}%
- **Improvement**: {impact['error_improvement']:.1f}% reduction in forecasting error

### Business Impact
- **Annual Cost Savings**: ${impact['annual_savings']:,.2f}
- **First Year ROI**: {impact['first_year_roi']:.1f}%
- **Subsequent Years ROI**: {impact['subsequent_years_roi']:.1f}%
- **Carbon Reduction**: {impact['annual_carbon_savings']:,.0f} kg CO₂ annually

## Key Insights
1. The most predictive features are previous consumption levels, particularly from previous days.
2. Strong daily and weekly seasonality patterns emerge in the consumption data.
3. Weather-based features would likely further improve the model (recommended for future work).

## Deployment Information
- Model is deployed as a REST API providing forecasts up to 30 days ahead.
- Monitoring system is in place to detect performance degradation.
- Recommended retraining schedule: Monthly, with data drift monitoring.

## Next Steps
1. Integrate weather forecast data to improve prediction accuracy.
2. Develop demand response strategies based on forecasts.
3. Extend the model to provide uncertainty estimates with prediction intervals.
"""

# Save the executive summary to a file
with open('reports/executive_summary.md', 'w') as f:
    f.write(executive_summary)

print("\nExecutive summary saved to 'reports/executive_summary.md'")

# Create a technical documentation template
technical_doc = """
# Energy Consumption Forecasting: Technical Documentation

## 1. Data Sources
- Daily electricity consumption data
- Time and date information
- Feature engineering approach for time features

## 2. Feature Engineering Process
- Created time-based features (hour, day of week, month, etc.)
- Implemented cyclical encoding for periodic features
- Generated lag features for autoregressive patterns
- Created rolling window statistics

## 3. Model Architecture
- Selected algorithm: XGBoost
- Key hyperparameters:
  - Learning rate: {learning_rate}
  - Max depth: {max_depth}
  - Number of estimators: {n_estimators}
  - Subsample ratio: {subsample}

## 4. Performance Metrics
- RMSE: {rmse}
- MAPE: {mape}%
- R²: {r2}

## 5. Deployment Architecture
- Flask REST API
- Endpoints:
  - /forecast: Get energy consumption forecasts
  - Parameters:
    - days: Number of days to forecast (1-30)

## 6. Monitoring System
- Daily performance tracking
- Alert thresholds:
  - RMSE > 5.0
  - MAPE > 10.0%

## 7. Maintenance Plan
- Retraining schedule: Monthly
- Data retention policy: 2 years rolling window
- Model versioning approach

## 8. Future Improvements
- Weather data integration
- Prediction intervals
- Customer segmentation models
- Demand response optimization
""".format(
    learning_rate=random_search.best_params_.get('learning_rate', 'N/A'),
    max_depth=random_search.best_params_.get('max_depth', 'N/A'),
    n_estimators=random_search.best_params_.get('n_estimators', 'N/A'),
    subsample=random_search.best_params_.get('subsample', 'N/A'),
    rmse=tuned_results['rmse'],
    mape=tuned_results['mape'],
    r2=tuned_results['r2']
)

# Save the technical documentation to a file
with open('reports/technical_documentation.md', 'w') as f:
    f.write(technical_doc)

print("Technical documentation saved to 'reports/technical_documentation.md'")
```

In [ ]:
# Generate an executive summary in markdown
executive_summary = f"""
# Energy Consumption Forecasting Project: Executive Summary

## Project Overview
This project developed a machine learning system to forecast daily electricity consumption
for a utility company. The model enables more accurate resource planning, price optimization,
and grid management.

## Key Results

### Model Performance
- **RMSE**: {tuned_results['rmse']:.2f} kilowatts
- **MAPE**: {tuned_results['mape']:.2f}%
- **Improvement**: {impact['error_improvement']:.1f}% reduction in forecasting error

### Business Impact
- **Annual Cost Savings**: ${impact['annual_savings']:,.2f}
- **First Year ROI**: {impact['first_year_roi']:.1f}%
- **Subsequent Years ROI**: {impact['subsequent_years_roi']:.1f}%
- **Carbon Reduction**: {impact['annual_carbon_savings']:,.0f} kg CO₂ annually

## Key Insights
1. The most predictive features are previous consumption levels, particularly from previous days.
2. Strong daily and weekly seasonality patterns emerge in the consumption data.
3. Weather-based features would likely further improve the model (recommended for future work).

## Deployment Information
- Model is deployed as a REST API providing forecasts up to 30 days ahead.
- Monitoring system is in place to detect performance degradation.
- Recommended retraining schedule: Monthly, with data drift monitoring.

## Next Steps
1. Integrate weather forecast data to improve prediction accuracy.
2. Develop demand response strategies based on forecasts.
3. Extend the model to provide uncertainty estimates with prediction intervals.
"""

In [ ]:
# Save the executive summary to a file
# with open('reports/executive_summary.md', 'w') as f:
with open('reports/executive_summary.md', 'w', encoding='utf-8') as f:
    f.write(executive_summary)

print("\nExecutive summary saved to 'reports/executive_summary.md'")

In [ ]:
# Create a technical documentation template
technical_doc = """
# Energy Consumption Forecasting: Technical Documentation

## 1. Data Sources
- Daily electricity consumption data
- Time and date information
- Feature engineering approach for time features

## 2. Feature Engineering Process
- Created time-based features (hour, day of week, month, etc.)
- Implemented cyclical encoding for periodic features
- Generated lag features for autoregressive patterns
- Created rolling window statistics

## 3. Model Architecture
- Selected algorithm: XGBoost
- Key hyperparameters:
  - Learning rate: {learning_rate}
  - Max depth: {max_depth}
  - Number of estimators: {n_estimators}
  - Subsample ratio: {subsample}

## 4. Performance Metrics
- RMSE: {rmse}
- MAPE: {mape}%
- R²: {r2}

## 5. Deployment Architecture
- Flask REST API
- Endpoints:
  - /forecast: Get energy consumption forecasts
  - Parameters:
    - days: Number of days to forecast (1-30)

## 6. Monitoring System
- Daily performance tracking
- Alert thresholds:
  - RMSE > 5.0
  - MAPE > 10.0%

## 7. Maintenance Plan
- Retraining schedule: Monthly
- Data retention policy: 2 years rolling window
- Model versioning approach

## 8. Future Improvements
- Weather data integration
- Prediction intervals
- Customer segmentation models
- Demand response optimization
""".format(
    learning_rate=random_search.best_params_.get('learning_rate', 'N/A'),
    max_depth=random_search.best_params_.get('max_depth', 'N/A'),
    n_estimators=random_search.best_params_.get('n_estimators', 'N/A'),
    subsample=random_search.best_params_.get('subsample', 'N/A'),
    rmse=tuned_results['rmse'],
    mape=tuned_results['mape'],
    r2=tuned_results['r2']
)

In [ ]:
# Save the technical documentation to a file
# with open('reports/technical_documentation.md', 'w') as f:
with open('reports/technical_documentation.md', 'w', encoding='utf-8') as f:
    f.write(technical_doc)

print("Technical documentation saved to 'reports/technical_documentation.md'")

## 11. Key Learnings and Takeaways

This end-to-end machine learning project for energy consumption forecasting demonstrates the complete ML lifecycle:

1. **Problem Definition**: We clearly defined the business problem and success metrics for energy consumption forecasting.

2. **Data Exploration**: We analyzed historic energy consumption data to understand patterns, trends, and seasonality.

3. **Feature Engineering**: The project showcased how to create effective time-based features, including:
   - Extracting calendar features (hour, day of week, month)
   - Creating cyclical features to handle periodic patterns
   - Building lag features to capture temporal dependencies
   - Generating rolling statistics to identify trends

4. **Model Development**: We compared multiple algorithms and found that tree-based ensemble methods (particularly XGBoost) performed best for this regression problem.

5. **Hyperparameter Tuning**: We used RandomizedSearchCV with TimeSeriesSplit to find optimal model parameters.

6. **Model Deployment**: We created a prediction function and REST API for making the model accessible to users.

7. **Monitoring and Maintenance**: We implemented performance tracking over time to detect model drift.

8. **Business Impact Analysis**: We quantified the financial and environmental benefits of the forecasting system.

Key takeaways from this project:

- **Time Series Specifics**: Time series forecasting requires specialized techniques including appropriate train/test splits, lag features, and evaluation methods.
- **Feature Engineering Importance**: Creating the right features is often more important than the choice of algorithm.
- **Model Explainability**: Understanding which features drive predictions is essential for building trust and improving the model.
- **Business Value**: Translating technical performance metrics (RMSE) to business outcomes (cost savings) is crucial for project success.

This project provides a template that can be adapted for various forecasting needs, from energy consumption to sales, website traffic, or other time series prediction problems.